# Databereinigung

Dieses Skript wurde zur Datenbereinigung der Eyetrackingdaten genutzt. 

In [3]:
## Einladen der genutzten Pakete ##
import pandas as pd
from pandas import DataFrame
import json
import numpy as np
import os
from collections import defaultdict
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf

In [14]:
## Einladen der Metadaten ##
meta = pd.read_csv('MetaData.csv', delimiter=";", header=0)
meta = meta.drop(['Unnamed: 18', 'Unnamed: 20'], axis=1)

#data.columns = ["a", "b", "c", "etc."]

In [15]:
## Erste Reinigungsstritte der Metadaten ##
meta= meta[meta.Turn != 6]
meta = meta.drop(['AudioFile','VideoFile', 'IntonationStart'], axis =1)

meta['Item-Nr.'] = meta['Dialogue'].astype(str) +'-'+meta['Turn'].astype(str)
meta=meta.dropna().reset_index(drop=True)
meta[['CrittUttStart', 'TurnStart','TurnEnd']] = meta[['CrittUttStart', 'TurnStart','TurnEnd']].apply(pd.to_numeric)

In [20]:
meta.describe()

,Dialogue,List,Turn,Condition,NoSent,Target,TurnStart,CrittUttStart,TurnEnd,RelevantStart
count,159.000000,159.000000,159.000000,159.000000,159.000000,159.000000,159.000000,159.000000,159.000000,159.000000
mean,4.522013,2.509434,3.012579,2.496855,2.232704,1.597484,9747.176101,12946.886792,14876.515723,14376.515723
std,2.288763,1.118701,1.414157,1.124380,0.648220,0.491954,7156.849985,7610.874794,7637.261242,7637.261242
min,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,63.000000,83.000000,2280.000000,1780.000000
25%,3.000000,2.000000,2.000000,1.500000,2.000000,1.000000,2898.000000,6355.000000,8737.000000,8237.000000
50%,5.000000,3.000000,3.000000,2.000000,2.000000,2.000000,10237.000000,12613.000000,14982.000000,14482.000000
75%,6.500000,3.500000,4.000000,3.500000,3.000000,2.000000,15590.500000,18918.000000,20684.000000,20184.000000
max,8.000000,4.000000,5.000000,4.000000,3.000000,2.000000,24087.000000,30263.000000,31872.000000,31372.000000


In [33]:
## Überprüfung der Länger der Zielsätze 
meta['CUlength'] = meta['TurnEnd']-meta['CrittUttStart']
meta['CUlength'].max()
meta['CUlength'].min()
meta['CUlength'].mean()

In [1]:
## Einladen der Eytetrackingdaten
results = pd.read_csv('full_results/Output/full_results.csv', delimiter="\t", header=0, dtype=None)

NameError: name 'pd' is not defined

In [17]:
## Übersicht über die Spalten und deren Datentypen
results.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12141032 entries, 0 to 12141031
Data columns (total 21 columns):
 #   Column                   Dtype  
---  ------                   -----  
 0   IP_INDEX                 object 
 1   IP_LABEL                 object 
 2   RECORDING_SESSION_LABEL  object 
 3   TRIAL_INDEX              int64  
 4   LEFT_GAZE_X              object 
 5   LEFT_GAZE_Y              object 
 6   LEFT_INTEREST_AREAS      object 
 7   RIGHT_GAZE_X             object 
 8   RIGHT_GAZE_Y             object 
 9   RIGHT_INTEREST_AREAS     object 
 10  TIMESTAMP                object 
 11  TRIAL_START_TIME         int64  
 12  ACCURACY                 int64  
 13  Click                    object 
 14  RT                       float64
 15  corr_answer              object 
 16  dialogue                 int64  
 17  endtime                  int64  
 18  itemtype                 object 
 19  list                     int64  
 20  question                 object 
dtypes: flo

In [19]:
## Erste Reinigungsstritte der Eytetrackingdaten
results = results.dropna()
results= results[results.dialogue != 9]
results = results.drop(['itemtype',], axis =1)

results=results.dropna().reset_index(drop=True)

In [20]:
## Umstellen der gemessenen Zeitpunkte
results["TIMESTAMP"] = [float(str(i).replace(",", ".")) for i in results["TIMESTAMP"]]
results['TIMESTAMP'] = pd.to_numeric(results['TIMESTAMP'])
results['Time'] = results['TIMESTAMP']-results['TRIAL_START_TIME']

In [21]:
## Umbenennen der Spalten
results.rename(columns={'RECORDING_SESSION_LABEL': 'Participant', 'dialogue': 'Dialogue', 'list': 'List', 'endtime': 'Endtime', 'corr_answer': 'Corr_answer', 'question': 'Question'}, inplace=True)

In [22]:
## Erster Ausschluss einzelner TN
results= results[results.Participant != '2006 (1)']
results= results[results.Participant != '1007 (1)']
results= results[results.Participant != '1008 (1)']
results= results[results.Participant != '3006 (1)'] 
results= results[results.Participant != '1002l']

In [23]:
## Auswahl jeder vierten Aufnahme, sodass ein Fixationswert alle vier Millisekunden (250Hz) übrig bleibt 
results = results.iloc[::4, :]
results=results.dropna().reset_index(drop=True)

## Verbindung beider Tabellen 

In [1]:
### Verbindung beider Tabelle (merging)
full_table = pd.merge(results, meta, on = ['Dialogue','List'])

NameError: name 'pd' is not defined

In [30]:
## Übersicht über die Spalten und deren Datentypen
full_table.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 11770000 entries, 0 to 11769999
Data columns (total 38 columns):
 #   Column                  Dtype  
---  ------                  -----  
 0   IP_INDEX                object 
 1   IP_LABEL                object 
 2   Participant             object 
 3   TRIAL_INDEX             int64  
 4   LEFT_GAZE_X             object 
 5   LEFT_GAZE_Y             object 
 6   LEFT_INTEREST_AREAS     object 
 7   RIGHT_GAZE_X            object 
 8   RIGHT_GAZE_Y            object 
 9   RIGHT_INTEREST_AREAS    object 
 10  TIMESTAMP               float64
 11  TRIAL_START_TIME        int64  
 12  ACCURACY                int64  
 13  Click                   object 
 14  RT                      float64
 15  Corr_answer             object 
 16  Dialogue                int64  
 17  Endtime                 int64  
 18  List                    int64  
 19  Question                object 
 20  Time                    float64
 21  Turn                    float

In [31]:
## Veränderung der Datentypen
full_table['Time'] = full_table['Time'].astype(int)
full_table['TurnEnd'] = full_table['TurnEnd'].astype(int)

In [36]:
## Zeitliche Beschränkung der Berücksichtigten Fixationen (für die )
final_table = full_table[(full_table['Time']>= full_table['TurnEnd']-500) & (full_table['Time']<= full_table['TurnEnd']+750)]
final_table=final_table.dropna().reset_index(drop=True)

In [37]:
final_table.describe()

,TRIAL_INDEX,TIMESTAMP,TRIAL_START_TIME,ACCURACY,RT,Dialogue,Endtime,List,Time,Turn,Condition,NoSent,Target,TurnStart,CrittUttStart,TurnEnd,RelevantStart,RelevantEnd,IntonationStart,CUlength
count,425337.000000,4.253370e+05,4.253370e+05,425337.000000,425337.000000,425337.000000,425337.000000,425337.000000,425337.000000,425337.000000,425337.000000,425337.000000,425337.000000,425337.000000,425337.000000,425337.000000,425337.000000,425337.000000,425337.000000,425337.000000
mean,5.499924,8.181804e+06,8.166875e+06,0.902564,3335.419023,4.499940,28942.979332,2.455890,14928.331516,2.999937,2.506587,2.224979,1.600028,9683.641672,12629.401933,14803.288094,14303.288094,15803.288094,14470.840063,-2173.886161
std,2.291326,7.735548e+06,7.735572e+06,0.296550,1630.777361,2.291321,3047.739015,1.117181,7662.150052,1.414226,1.123913,0.651458,0.489893,7152.648745,7643.930921,7653.642493,7653.642493,7653.642493,7632.215650,2103.082705
min,2.000000,1.600220e+05,1.582410e+05,0.000000,0.000000,1.000000,25000.000000,1.000000,1780.000000,1.000000,1.000000,1.000000,1.000000,63.000000,83.000000,2280.000000,1780.000000,3280.000000,1910.000000,-20464.000000
25%,3.000000,1.665308e+06,1.655495e+06,1.000000,2205.110126,2.000000,27000.000000,1.000000,8392.000000,2.000000,1.000000,2.000000,1.000000,2757.000000,6184.000000,7789.000000,7289.000000,8789.000000,7307.000000,-2197.000000
50%,5.000000,5.093644e+06,5.091588e+06,1.000000,2906.872025,4.000000,28000.000000,2.000000,14786.000000,3.000000,2.000000,2.000000,2.000000,9473.000000,12463.000000,14982.000000,14482.000000,15982.000000,14680.000000,-1839.000000
75%,8.000000,1.257639e+07,1.255373e+07,1.000000,4082.574066,6.000000,31000.000000,3.000000,20835.000000,4.000000,4.000000,3.000000,2.000000,15381.000000,18879.000000,20214.000000,19714.000000,21214.000000,20016.000000,-1586.000000
max,9.000000,2.456232e+07,2.453429e+07,1.000000,9997.702208,8.000000,36000.000000,4.000000,32622.000000,5.000000,4.000000,3.000000,2.000000,24087.000000,30263.000000,31872.000000,31372.000000,32872.000000,31529.000000,-1208.000000


In [38]:
## Übersicht über die Spalten und deren Datentypen
final_table.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 425337 entries, 2445 to 11760979
Data columns (total 38 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   IP_INDEX                425337 non-null  object 
 1   IP_LABEL                425337 non-null  object 
 2   Participant             425337 non-null  object 
 3   TRIAL_INDEX             425337 non-null  int64  
 4   LEFT_GAZE_X             425337 non-null  object 
 5   LEFT_GAZE_Y             425337 non-null  object 
 6   LEFT_INTEREST_AREAS     425337 non-null  object 
 7   RIGHT_GAZE_X            425337 non-null  object 
 8   RIGHT_GAZE_Y            425337 non-null  object 
 9   RIGHT_INTEREST_AREAS    425337 non-null  object 
 10  TIMESTAMP               425337 non-null  float64
 11  TRIAL_START_TIME        425337 non-null  int64  
 12  ACCURACY                425337 non-null  int64  
 13  Click                   425337 non-null  object 
 14  RT             

In [40]:
## Auswahl der Spalten für die fertige Tabelle
table_subset = final_table[["Participant", "TRIAL_INDEX", "ACCURACY", "Click", "RT", "TurnEnd","Corr_answer", "List", "Time", \
                            "Turn", "Condition", "Target","SentType", "NoSent", "Gaze", "LEFT_INTEREST_AREAS", \
                            "RIGHT_INTEREST_AREAS", "Item-Nr."]]
table_subset = table_subset.rename(columns={"Time": "ItemTime", "Item-Nr.": "ItemNr"})

In [41]:
final_table.columns.values

array(['IP_INDEX', 'IP_LABEL', 'Participant', 'TRIAL_INDEX',
       'LEFT_GAZE_X', 'LEFT_GAZE_Y', 'LEFT_INTEREST_AREAS',
       'RIGHT_GAZE_X', 'RIGHT_GAZE_Y', 'RIGHT_INTEREST_AREAS',
       'TIMESTAMP', 'TRIAL_START_TIME', 'ACCURACY', 'Click', 'RT',
       'Corr_answer', 'Dialogue', 'Endtime', 'List', 'Question', 'Time',
       'Turn', 'Condition', 'SentType', 'NoSent', 'Target', 'Gaze',
       'TurnStart', 'CrittUttStart', 'TurnEnd', 'RelevantStart',
       'RelevantEnd', 'IntonationOrt', 'Intonation', 'IntonationStart',
       'Intonation from Offset', 'Item-Nr.', 'CUlength'], dtype=object)

In [44]:
final_table.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 425337 entries, 0 to 425336
Data columns (total 38 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   IP_INDEX                425337 non-null  object 
 1   IP_LABEL                425337 non-null  object 
 2   Participant             425337 non-null  object 
 3   TRIAL_INDEX             425337 non-null  int64  
 4   LEFT_GAZE_X             425337 non-null  object 
 5   LEFT_GAZE_Y             425337 non-null  object 
 6   LEFT_INTEREST_AREAS     425337 non-null  object 
 7   RIGHT_GAZE_X            425337 non-null  object 
 8   RIGHT_GAZE_Y            425337 non-null  object 
 9   RIGHT_INTEREST_AREAS    425337 non-null  object 
 10  TIMESTAMP               425337 non-null  float64
 11  TRIAL_START_TIME        425337 non-null  int64  
 12  ACCURACY                425337 non-null  int64  
 13  Click                   425337 non-null  object 
 14  RT                  

In [48]:
#table_subset["CorrectClick"] = np.where(table_subset['Click']==table_subset['Corr_answer'], 1,0)
table_subset.loc[table_subset['Click'] == table_subset['Corr_answer'], 'CorrectClick'] = 0
table_subset.loc[table_subset['Click'] != table_subset['Corr_answer'], 'CorrectClick'] = 1

In [49]:
## Zuordnung zu AoI
table_subset['LEFT_INTEREST_AREAS'] = table_subset['LEFT_INTEREST_AREAS'].replace({"[ 2 ]": 2, "[ 1 ]": 1, '[]': 0})
table_subset['RIGHT_INTEREST_AREAS'] = table_subset['RIGHT_INTEREST_AREAS'].replace({"[ 2 ]": 2, "[ 1 ]": 1, '[]': 0})
table_subset['Aoi'] = table_subset['LEFT_INTEREST_AREAS']+table_subset['RIGHT_INTEREST_AREAS']

In [50]:
## Zuordnung der AoI
table_subset.loc[table_subset['Aoi'] != table_subset['Target'], 'CorrectLooks'] = 0
table_subset.loc[table_subset['Aoi'] == table_subset['Target'], 'CorrectLooks'] = 1

In [51]:
## Anpassen des Datentyps und Zurodnung zur Zielfigur
table_subset['Aoi'] = table_subset['Aoi'].astype(str)
table_subset['Aoi'] = 'Figure_'+table_subset['Aoi']

In [52]:
## Anpassung der Zielfigur und der sprechenden Figur
table_subset.loc[table_subset['Aoi'] == 'Figure_1',  'Figure_1'] = 1
table_subset.loc[table_subset['Aoi'] != 'Figure_1',  'Figure_1'] = 0
table_subset.loc[table_subset['Aoi'] == 'Figure_2',  'Figure_2'] = 1
table_subset.loc[table_subset['Aoi'] != 'Figure_2',  'Figure_2'] = 0
table_subset['Figure_2'] = table_subset['Figure_2'].astype(int)
table_subset['Figure_1'] = table_subset['Figure_1'].astype(int)
table_subset['Figure_2'] = table_subset['Figure_2'].astype(str)
table_subset['Figure_1'] = table_subset['Figure_1'].astype(str)

In [53]:
table_subset.loc[table_subset['Aoi'] == 'Figure_0', 'Trackloss'] = True
table_subset.loc[table_subset['Aoi'] == 'Figure_1', 'Trackloss'] = False
table_subset.loc[table_subset['Aoi'] == 'Figure_2', 'Trackloss'] = False

In [54]:
table_subset['Trackloss'].unique()

array([False, True], dtype=object)

In [55]:
#table_subset['Trackloss'] = table_subset['Trackloss'].astype(int)
table_subset['CorrectLooks'] = table_subset['CorrectLooks'].astype(int)
table_subset['CorrectClick'] = table_subset['CorrectClick'].astype(int)

In [56]:
table_subset = table_subset.drop(columns=['LEFT_INTEREST_AREAS', 'RIGHT_INTEREST_AREAS'])

In [57]:
table_subset = table_subset.drop(columns=['Corr_answer', 'Click'])

In [58]:
table_subset

,Participant,TRIAL_INDEX,ACCURACY,RT,TurnEnd,List,ItemTime,Turn,Condition,Target,SentType,NoSent,Gaze,ItemNr,CorrectClick,Aoi,CorrectLooks,Figure_1,Figure_2,Trackloss
0,1001,2,1,1590.415166,2454,1,1956,1.0,4.0,2.0,question,1.0,yes,1.0-1.0,0,Figure_2,1,0,1,False
1,1001,2,1,1590.415166,2454,1,1960,1.0,4.0,2.0,question,1.0,yes,1.0-1.0,0,Figure_2,1,0,1,False
2,1001,2,1,1590.415166,2454,1,1964,1.0,4.0,2.0,question,1.0,yes,1.0-1.0,0,Figure_2,1,0,1,False
3,1001,2,1,1590.415166,2454,1,1968,1.0,4.0,2.0,question,1.0,yes,1.0-1.0,0,Figure_2,1,0,1,False
4,1001,2,1,1590.415166,2454,1,1972,1.0,4.0,2.0,question,1.0,yes,1.0-1.0,0,Figure_2,1,0,1,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
425332,4008,5,1,2409.160945,22706,4,23438,5.0,1.0,2.0,declarative,3.0,no,3.0-5.0,0,Figure_1,0,1,0,False
425333,4008,5,1,2409.160945,22706,4,23442,5.0,1.0,2.0,declarative,3.0,no,3.0-5.0,0,Figure_1,0,1,0,False
425334,4008,5,1,2409.160945,22706,4,23446,5.0,1.0,2.0,declarative,3.0,no,3.0-5.0,0,Figure_1,0,1,0,False
425335,4008,5,1,2409.160945,22706,4,23450,5.0,1.0,2.0,declarative,3.0,no,3.0-5.0,0,Figure_1,0,1,0,False


### TurnTime berechnen. 
*  jeder Turn beginnt bei 0 msec und dauert dann die Gesamtlänge des Turns. --> TurnTime
* TurnTime nach Offset normalisiern
    * kürzeste Critical Utterance 1401msec; da wir aber die Antizipation des Turnendes ansehen wollen, kürzen wir den Timestart auf -500ms -> TimeStart
    * Maximales Zeitfenster bis +750 msec ab Offset

In [59]:
table_subset['Turn'] = table_subset['Turn'].astype(str)

In [60]:
## Normalisierung der Zeit mit dem Turnende als Nullpunkt
table_subset["TurnWindow"] = np.where(((table_subset['ItemTime'] >= (table_subset['TurnEnd']-1400)) & 
    (table_subset['ItemTime'] <= (table_subset['TurnEnd']+1000))), table_subset['ItemNr']+'-'+table_subset['Turn'], 0)
table_subset["TurnTime"] = np.where(table_subset["TurnWindow"] != 0, table_subset["ItemTime"]-table_subset['TurnEnd'], 0)

In [62]:
table_subset['TurnTime']

0        -498
1        -494
2        -490
3        -486
4        -482
         ... 
425332    732
425333    736
425334    740
425335    744
425336    748
Name: TurnTime, Length: 425337, dtype: int32

In [87]:
## Speicherung der bereinigten und für die Analyse vorbereiteten Tabelle
table_subset.to_csv(r'cleaned_table.txt', header=1, index=None, sep='\t')